# 🏺 Tumulus LiDAR Detector — interactive demo

**Find ancient burial mounds (tumuli) in airborne LiDAR — right in your browser, nothing to install.**

It downloads public **0.5 m LiDAR** terrain for an area you choose, turns it into a relief image, and runs a small neural network that flags dome-shaped mounds.

---

## ▶️ Quick start
Press **Runtime → Run all** (top menu) and wait ~1–3 minutes. It scans a default mound field in southern Romania and shows the results.

## 🗺️ Scan your own spot
In **Step 2** a map of Romania appears. The **blue area is where 0.5 m LiDAR exists** — the only place this demo can scan. **Click inside the blue** to drop a pin (or drag it), then run **Step 3**.

> ⚠️ **Coverage is the blue area only** (ANCPI LAKI III, mostly Oltenia / SW Romania). A pin outside the blue has no LiDAR and returns no map — the notebook tells you if so.

## 📦 What you get
- a **relief heatmap** of the scanned area, mounds circled in green;
- a **close-up** of each detected mound;
- a **table** of exact coordinates + a one-click Google Maps link;
- a **CSV** you can download.

## ⏱️ How long
The default **2 km** box takes ~1–3 min, mostly a one-time tile download (cached after). Bigger boxes take proportionally longer (a 5 km box was ~15 min on free Colab). The model is tiny, so a GPU barely helps — CPU is fine.

---
Repo: https://github.com/ObuObuHub/tumulus-lidar-detector · *Shape is not proof — only fieldwork confirms a tumulus.*

In [ ]:
# STEP 1 — Set up (run once): download the code + model and the few extra libraries.
# torch, numpy and Pillow already ship with Colab.
!git clone -q https://github.com/ObuObuHub/tumulus-lidar-detector.git
%cd tumulus-lidar-detector
!pip install -q pyproj ipyleaflet
print("\n✅ Ready. Run Step 2 to pick a location on the map.")

## Step 2 — Pick a location
Click inside the **blue** (0.5 m LiDAR coverage) to drop your pin, or drag it — the **lat / lon boxes fill in automatically**. You can also type coordinates into the boxes to move the pin. The default pin sits on a known mound field. Then run **Step 3**.

In [ ]:
# STEP 2 — Pick where to scan: click the map (or drag the pin), and the lat/lon boxes
# below fill in automatically. You can also type into the boxes to move the pin.
# Blue = the area with 0.5 m LiDAR, where this demo can scan. Stay inside the blue.
import json
from IPython.display import display

PIN = {'lat': 44.043, 'lon': 23.522}   # default: a known mound field, inside coverage
_g = {'busy': False}                    # re-entrancy guard for the two-way sync

# Coordinate boxes — these work even if the map widget does not.
try:
    from ipywidgets import FloatText, HBox, HTML
    lat_box = FloatText(value=PIN['lat'], description='lat:', step=0.0001)
    lon_box = FloatText(value=PIN['lon'], description='lon:', step=0.0001)
    _info = HTML()
    _have_widgets = True
except Exception as _e:
    _have_widgets = False
    print("Widgets unavailable (%s) — set coordinates in Step 3." % _e)

def _set_label():
    if _have_widgets:
        _info.value = ("📍 <b>Pin:</b> lat <code>%.5f</code>, lon <code>%.5f</code>"
                       " &nbsp;—&nbsp; now run <b>Step 3</b> to scan." % (PIN['lat'], PIN['lon']))

# Try to add the interactive map (optional — boxes are the fallback).
_map_ok = False
if _have_widgets:
    try:
        from google.colab import output as _co
        _co.enable_custom_widget_manager()
    except Exception:
        pass
    try:
        from ipyleaflet import Map, GeoJSON, Marker, basemaps
        _cov = json.load(open('assets/laki3_coverage.geojson'))
        _m = Map(center=(44.7, 23.3), zoom=7, scroll_wheel_zoom=True,
                 basemap=basemaps.OpenStreetMap.Mapnik)
        _m.add(GeoJSON(data=_cov, name='0.5 m LiDAR coverage',
                       style={'color': '#1d4ed8', 'fillColor': '#2563eb',
                              'fillOpacity': 0.30, 'weight': 1}))
        _mk = Marker(location=(PIN['lat'], PIN['lon']), draggable=True, title='Scan here')
        _m.add(_mk)

        def _boxes_from_pin():
            _g['busy'] = True
            lat_box.value, lon_box.value = PIN['lat'], PIN['lon']
            _g['busy'] = False
            _set_label()

        def _on_click(**kw):
            if kw.get('type') == 'click':
                la, lo = kw['coordinates']
                PIN['lat'], PIN['lon'] = round(la, 5), round(lo, 5)
                _g['busy'] = True; _mk.location = (PIN['lat'], PIN['lon']); _g['busy'] = False
                _boxes_from_pin()
        _m.on_interaction(_on_click)

        def _on_drag(_change):
            if _g['busy']:
                return
            PIN['lat'], PIN['lon'] = round(_mk.location[0], 5), round(_mk.location[1], 5)
            _boxes_from_pin()
        _mk.observe(_on_drag, 'location')
        _map_ok = True
    except Exception as _e:
        print("Map unavailable (%s) — use the lat/lon boxes below." % _e)

# Boxes -> pin (and PIN), with guard so the two-way sync can't loop.
if _have_widgets:
    def _on_box(_change):
        if _g['busy']:
            return
        PIN['lat'], PIN['lon'] = float(lat_box.value), float(lon_box.value)
        if _map_ok:
            _g['busy'] = True; _mk.location = (PIN['lat'], PIN['lon']); _g['busy'] = False
        _set_label()
    lat_box.observe(_on_box, 'value'); lon_box.observe(_on_box, 'value')
    _set_label()
    if _map_ok:
        display(_m)
    display(HBox([lat_box, lon_box]), _info)
    print("Click the map (or edit the boxes) to set your point, then run Step 3.")

## Step 3 — Scan
Pick a box size and run. It scans wherever the pin / lat-lon boxes from Step 2 point.

In [ ]:
# STEP 3 — Scan the pinned area.
area_km = 2  #@param {type:"slider", min:1, max:8, step:1}

import json
try:
    lon, lat = PIN['lon'], PIN['lat']
except NameError:
    lat, lon = 44.043, 23.522   # default if Step 2 was skipped

def _inside(lon, lat):
    # point-in-polygon over the 0.5 m coverage rectangles
    cov = json.load(open('assets/laki3_coverage.geojson'))
    for f in cov['features']:
        ring = f['geometry']['coordinates'][0]
        c = False; n = len(ring); j = n - 1
        for i in range(n):
            xi, yi = ring[i]; xj, yj = ring[j]
            if ((yi > lat) != (yj > lat)) and (lon < (xj - xi) * (lat - yi) / (yj - yi) + xi):
                c = not c
            j = i
        if c:
            return True
    return False

print("Scan point: lat %s, lon %s  |  box %s km" % (lat, lon, area_km))
if not _inside(lon, lat):
    print("\n⚠️  This point is OUTSIDE the 0.5 m LiDAR coverage (the blue area on the map).")
    print("    The scan would find no tiles. Move the pin into the blue and run again.")
else:
    !python tools/scan_zone.py {lon} {lat} {area_km}

### 🟢 Results — relief map & detected mounds
**Green circles** = kept candidates (probable mounds). **Grey ×** = shapes the filters rejected. Below the map, a close-up of each kept candidate (shown only if at least one was found).

In [ ]:
import os
from IPython.display import Image, display
if os.path.exists('review/zone_view.jpg'):
    display(Image('review/zone_view.jpg'))           # heatmap of the scanned area
    if os.path.exists('review/zone_board.jpg'):
        display(Image('review/zone_board.jpg'))       # close-up crops (only when candidates kept)
    else:
        print('No candidates kept in this area - no close-up board. The heatmap above is the LiDAR terrain.')
else:
    print('No map produced: the scan found no LiDAR tiles. Coverage is the blue area only (LAKI III). Move the pin into the blue (Step 2) and re-run Step 3.')

### 📋 Coordinates, metrics & CSV download
Each detected mound, highest confidence first.

| column | meaning |
|---|---|
| `score` | detector confidence (0–1) |
| `coh` | directional coherence — *lower* = more dome-like (less linear) |
| `pgate` | shape-filter probability — *higher* = smoother, rounder dome |
| `map` | opens the point in Google Maps |

In [ ]:
import os, pandas as pd
from IPython.display import HTML, display

if not os.path.exists('/tmp/zone_dets.csv'):
    print('No detections file - the scan produced no output (pin outside coverage, or a tile-download hiccup). See Step 3.')
else:
    det = pd.read_csv('/tmp/zone_dets.csv')
    mounds = det[det['keep'] == 1][['lon', 'lat', 'score', 'coh', 'pgate']].sort_values('score', ascending=False).reset_index(drop=True)
    mounds.insert(0, 'id', range(1, len(mounds) + 1))
    if len(mounds) > 0:
        print(f'{len(mounds)} probable mound(s) detected - coordinates + detection metrics (highest score first):')
        shown = mounds.copy()
        shown['map'] = shown.apply(lambda r: f'<a href="https://www.google.com/maps?q={r.lat},{r.lon}" target="_blank">view</a>', axis=1)
        display(HTML(shown.to_html(escape=False, index=False)))
        mounds.to_csv('detected_mounds.csv', index=False)   # id, lon, lat, score, coh, pgate
        print('Saved detected_mounds.csv (id, lon, lat, score, coh, pgate).')
        try:
            from google.colab import files
            files.download('detected_mounds.csv')           # triggers a browser download
        except Exception:
            print('detected_mounds.csv saved - see the Files panel on the left.')
    else:
        print('Scan completed: 0 mounds matched tumulus form in this area, which is normal for most points (the model is selective). Nothing to download.')

---
**Scan another place:** move the pin in Step 2 (or set coordinates in Step 3) and re-run Step 3 onward — keep it inside the blue coverage.

**Check accuracy against your own mounds:** run `tools/benchmark.py your_truth.csv combined_cnn.pt` with a CSV of `lon,lat`. Romanian site coordinates are withheld here for looting safety — bring your own confirmed set, or a public one.

*Built by [ObuObuHub](https://github.com/ObuObuHub/tumulus-lidar-detector). Shape is not proof — only fieldwork confirms a tumulus.*